In [ ]:
!pip install langchain-community
import os
import numpy as np
from langchain_community.embeddings import HuggingFaceBgeEmbeddings, OpenAIEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFaceEndpoint
from langchain_huggingface.llms import HuggingFacePipeline



In [ ]:
!pip install langchain-huggingface

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("resume.zip", 'r') as zip_ref:
    zip_ref.extractall("resume")  # extracts files inside "resume" folder

In [ ]:
import os

print("Root files/folders:", os.listdir())
print("Files in shown_resumes:", os.listdir("resume"))

In [ ]:
from urllib.request import urlretrieve

In [ ]:
files = [
   "file:///C:/Users/admin/Downloads/Resumes_Part1%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part2.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part3.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part4.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part6%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part7.pdf",
   "file:///C:/Users/admin/Downloads/Resume_8.pdf",
   "file:///C:/Users/admin/Downloads/Resume_9.pdf",
   "file:///C:/Users/admin/Downloads/Resume_10.pdf",
   "file:///C:/Users/admin/Downloads/Resume_10.pdf",
   "file:///C:/Users/admin/Downloads/Resume_13.pdf",
   "file:///C:/Users/admin/Downloads/Resume_14%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resume_15.pdf",
]
os.makedirs('resume',exist_ok=True)

In [ ]:
import os
import fitz

pdf_dir = "resume"
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

for file in pdf_files:
    file_path = os.path.join(pdf_dir, file)
    doc = fitz.open(file_path)  # <-- Use this relative path only
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()

    print(f"\nExtracted text from {file}:\n{text[:300]}...\n")

In [ ]:
!pip install pymupdf

In [ ]:
loader=PyPDFDirectoryLoader('resume')

In [ ]:
docs_before_split=loader.load()

In [ ]:
!pip install pypdf

In [ ]:
len(docs_before_split[0].page_content)

In [ ]:
import os

pdf_dir = "resumes"
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
print("PDF files found:", pdf_files)

In [ ]:
from langchain.document_loaders import PyPDFLoader
import os

folder = "resume"
docs_before_split = []

for filename in os.listdir(folder):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(folder, filename)  # use filename with %20 as is
        print(f"Loading: {path}")
        loader = PyPDFLoader(path)
        docs = loader.load()
        docs_before_split.extend(docs)
print(f"Total pages loaded: {docs_before_split}")

In [ ]:
text_splitter =  RecursiveCharacterTextSplitter(
    chunk_size =170,
    chunk_overlap = 20
)
docs_after_split = text_splitter.split_documents(docs_before_split)

In [ ]:
len(docs_after_split[0].page_content)

In [ ]:
avg_doc_length = lambda docs: sum([len(doc.page_content) for doc in docs])//len(docs)

In [ ]:
avg_char_before_split = avg_doc_length(docs_before_split)
avg_char_after_split = avg_doc_length(docs_after_split)

In [ ]:
print(f'before split: {avg_char_before_split}')
print(f'after split: {avg_char_after_split}')

In [ ]:
huggingface_embedings=HuggingFaceBgeEmbeddings(
    model_name= "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs= {'device':'cpu'},
    encode_kwargs= {"normalize_embedings": True}
)

In [ ]:
vector_store=FAISS.from_documents(docs_after_split,huggingface_embedings)

In [ ]:
query="Find candidates with TensorFlow + AWS experience?"

In [ ]:
relevant_document=vector_store.similarity_search(query)

In [ ]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={'k':3})

In [ ]:
access_token="****"


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

def setup_llm():
    model_id = "google/flan-t5-small"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=128,
        do_sample=False,
    )
    return pipe

In [ ]:
from langchain.prompts import PromptTemplate
prompt_template = """Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

{context}

Question: {question}

Helpful Answer:
"""

PROMPT = PromptTemplate(
 template=prompt_template, input_variables=["context", "question"]
)

In [ ]:
pip install -U langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFacePipeline


In [ ]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

# Create the pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)

# Wrap it
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
pipe = setup_llm()  # your HuggingFace pipeline (Text2TextGenerationPipeline)
llm = HuggingFacePipeline(pipeline=pipe)  # wrap it as a LangChain Runnable

In [ ]:
from langchain.chains import RetrievalQA

retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

USE the RAG

In [ ]:
result = retrievalQA.invoke({"query" : query})
print(result)